# Sentinel-1 EW L0 RAW Scene Search

Searches the CDSE catalogue for Sentinel-1 EW Level-0 RAW products intersecting an AOI and date
range, and writes the matching scene names to a text file for `L0_PRODUCTS` in
[`batch_order_l0_to_slc_and_upload.ipynb`](./batch_order_l0_to_slc_and_upload.ipynb).

**Steps:** authenticate → define AOI (GeoJSON or hardcoded fallback) → search the catalogue →
write the scene list.

[OData API docs](https://documentation.dataspace.copernicus.eu/APIs/OData.html)

## Configuration

Set the AOI, date range, and output path below. Credentials come from
`CDSE_USERNAME`/`CDSE_PASSWORD` env vars (or enter directly).

**AOI**: set `GEOJSON_PATH` to a `.geojson`/`.json` file (features are unioned into one search
area), or leave `None` to use `HARDCODED_GEOMETRY_WKT` below.

In [ ]:
import os
from dotenv import load_dotenv

try:
    # load credentials from root .env file
    load_dotenv("../../.env")
except:
    print("Could not find .env file with credentials.")

# --- INPUT -------------------------------------------------------------------
# GeoJSON file defining the AOI, or None to use HARDCODED_GEOMETRY_WKT below.
GEOJSON_PATH = None  # e.g. "./aoi.geojson"

# Off the shackleton ice shelf - replace with AOI
HARDCODED_GEOMETRY_WKT = "POLYGON ((85 -60, 105 -60, 105 -50, 85 -50, 85 -60))"

# Search date range (ContentDate/Start), inclusive, as YYYY-MM-DD
START_DATE = "2026-03-01"
END_DATE = "2026-04-01"

# Cap on number of results returned (None for no cap)
MAX_RESULTS = None

# Where to write the resulting scene name list
OUTPUT_LIST_PATH = "EW_L0_shackleton_BOM_scene_list.txt"

# Credentials — prefer env vars so they are not committed
CDSE_USERNAME = os.environ.get("CDSE_LOGIN", "")  # or set directly: "your@email.com"
CDSE_PASSWORD = os.environ.get("CDSE_PASSWORD", "")  # or set directly: "yourpassword"
# -----------------------------------------------------------------------------

CATALOGUE_URL = "https://catalogue.dataspace.copernicus.eu/odata/v1"
TOKEN_URL = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"

print(f"GeoJSON path : {GEOJSON_PATH}")
print(f"Date range   : {START_DATE} to {END_DATE}")
print(f"Output path  : {os.path.abspath(OUTPUT_LIST_PATH)}")

## 1. Authenticate

In [ ]:
import requests
import getpass
import time as _time

if not CDSE_USERNAME:
    CDSE_USERNAME = input("Copernicus username (email): ")
if not CDSE_PASSWORD:
    CDSE_PASSWORD = getpass.getpass("Copernicus password: ")


def get_token(username: str, password: str) -> str:
    resp = requests.post(
        TOKEN_URL,
        data={
            "client_id": "cdse-public",
            "username": username,
            "password": password,
            "grant_type": "password",
        },
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        timeout=30,
    )
    if not resp.ok:
        try:
            detail = resp.json()
        except Exception:
            detail = resp.text
        raise RuntimeError(
            f"Authentication failed ({resp.status_code}): {detail}\n\n"
            "Check that:\n"
            "  1. Your Copernicus Data Space account is verified at https://dataspace.copernicus.eu\n"
            "  2. You are using your registration email + password (not a Google/GitHub SSO login)\n"
            "  3. There are no leading/trailing spaces in your credentials"
        )
    data = resp.json()
    return data["access_token"], data.get("expires_in", 600)


TOKEN, TOKEN_EXPIRES_IN = get_token(CDSE_USERNAME, CDSE_PASSWORD)
TOKEN_ACQUIRED_AT = _time.monotonic()


def auth_headers() -> dict:
    """Return headers with a fresh token, refreshing if within 60s of expiry."""
    global TOKEN, TOKEN_ACQUIRED_AT, TOKEN_EXPIRES_IN
    if _time.monotonic() - TOKEN_ACQUIRED_AT > (TOKEN_EXPIRES_IN - 60):
        print("Refreshing token...")
        TOKEN, TOKEN_EXPIRES_IN = get_token(CDSE_USERNAME, CDSE_PASSWORD)
        TOKEN_ACQUIRED_AT = _time.monotonic()
    return {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}


print(f"Authenticated successfully. Token valid for {TOKEN_EXPIRES_IN}s.")

## 2. Define the Area of Interest

Loads `GEOJSON_PATH` if set (unioning features, reprojecting to EPSG:4326 if needed), otherwise
parses `HARDCODED_GEOMETRY_WKT`. Builds the CDSE catalogue's
`OData.CSC.Intersects(Footprint=geography'SRID=4326;<WKT>')` filter from the result.

In [ ]:
import geopandas as gpd
from shapely import wkt as shapely_wkt

if GEOJSON_PATH and os.path.exists(GEOJSON_PATH):
    aoi_gdf = gpd.read_file(GEOJSON_PATH)
    if aoi_gdf.crs is not None and aoi_gdf.crs.to_epsg() != 4326:
        aoi_gdf = aoi_gdf.to_crs(epsg=4326)
    aoi_geometry = aoi_gdf.geometry.union_all()
    print(f"Loaded AOI from {GEOJSON_PATH} ({len(aoi_gdf)} feature(s)), unioned into one geometry.")
else:
    if GEOJSON_PATH:
        print(f"GEOJSON_PATH '{GEOJSON_PATH}' not found — falling back to HARDCODED_GEOMETRY_WKT.")
    aoi_geometry = shapely_wkt.loads(HARDCODED_GEOMETRY_WKT)

AOI_WKT = aoi_geometry.wkt
GEOMETRY_LITERAL = f"geography'SRID=4326;{AOI_WKT}'"

print(f"AOI bounds (minx, miny, maxx, maxy): {aoi_geometry.bounds}")
print(f"AOI WKT length: {len(AOI_WKT)} chars")
if len(AOI_WKT) > 8000:
    print(
        "WARNING: AOI WKT is very long and may exceed URL length limits in the search "
        "request below. Consider simplifying the geometry, e.g. "
        "aoi_geometry = aoi_geometry.simplify(0.01)"
    )

## 3. Search the CDSE Catalogue

Queries `Products` filtered on Sentinel-1, `_EW_RAW__0` in the name (EW Level-0 RAW naming
convention), the date range, and the AOI footprint. Offline (LTA) scenes are included — the
batch notebook handles LTA retrieval. Pagination follows `@odata.nextLink`.

In [ ]:
def build_filter(geometry_literal: str, start_date: str, end_date: str) -> str:
    return (
        "Collection/Name eq 'SENTINEL-1' "
        "and contains(Name,'_EW_RAW__0') "
        f"and ContentDate/Start ge {start_date}T00:00:00.000Z "
        f"and ContentDate/Start le {end_date}T23:59:59.999Z "
        f"and OData.CSC.Intersects(Footprint={geometry_literal})"
    )


def search_ew_raw_scenes(
    geometry_literal: str,
    start_date: str,
    end_date: str,
    max_results: int | None = None,
    page_size: int = 100,
) -> list[dict]:
    filter_str = build_filter(geometry_literal, start_date, end_date)
    url = f"{CATALOGUE_URL}/Products"
    params = {
        "$filter": filter_str,
        "$orderby": "ContentDate/Start asc",
        "$top": page_size,
    }

    results = []
    while url:
        resp = requests.get(url, params=params, headers=auth_headers(), timeout=60)
        resp.raise_for_status()
        data = resp.json()
        page = data.get("value", [])
        results.extend(page)
        print(f"  ...{len(results)} product(s) so far")

        if max_results and len(results) >= max_results:
            results = results[:max_results]
            break

        # @odata.nextLink already carries the full query string, so params must be dropped
        url = data.get("@odata.nextLink")
        params = None

    return results


print("Searching CDSE catalogue for EW L0 RAW products...")
ew_raw_products = search_ew_raw_scenes(GEOMETRY_LITERAL, START_DATE, END_DATE, max_results=MAX_RESULTS)
print(f"\nFound {len(ew_raw_products)} matching product(s).")

## 4. Write the Scene List

Prints a preview table, writes scene names to `OUTPUT_LIST_PATH` (one per line), and prints a
Python list literal ready to paste into `L0_PRODUCTS`.

In [ ]:
import json
import os

print(f"{'Name':<75s} {'ContentDate/Start':<25s} {'Online':<6s}")
for p in ew_raw_products:
    content_start = (p.get("ContentDate") or {}).get("Start", "?")
    print(f"{p.get('Name', '?'):<75s} {content_start:<25s} {str(p.get('Online')):<6s}")

EW_RAW_SCENES = [p["Name"] for p in ew_raw_products]

with open(OUTPUT_LIST_PATH, "w") as f:
    f.write("\n".join(EW_RAW_SCENES) + "\n")

print(f"\nWrote {len(EW_RAW_SCENES)} scene name(s) to {os.path.abspath(OUTPUT_LIST_PATH)}")

# --- Also write a GeoJSON with the same base filename ---
OUTPUT_GEOJSON_PATH = os.path.splitext(OUTPUT_LIST_PATH)[0] + ".geojson"

features = []
for p in ew_raw_products:
    geom = p.get("GeoFootprint")
    if geom is None:
        continue
    content_date = p.get("ContentDate") or {}
    feature = {
        "type": "Feature",
        "geometry": geom,
        "properties": {
            "Name": p.get("Name"),
            "Id": p.get("Id"),
            "ContentLength": p.get("ContentLength"),
            "Online": p.get("Online"),
            "OriginDate": p.get("OriginDate"),
            "PublicationDate": p.get("PublicationDate"),
            "ContentDate_Start": content_date.get("Start"),
            "ContentDate_End": content_date.get("End"),
            "S3Path": p.get("S3Path"),
        },
    }
    features.append(feature)

feature_collection = {
    "type": "FeatureCollection",
    "features": features,
}

with open(OUTPUT_GEOJSON_PATH, "w") as f:
    json.dump(feature_collection, f, indent=2)

print(f"Wrote {len(features)} feature(s) to {os.path.abspath(OUTPUT_GEOJSON_PATH)}")

print("\nPython list literal (paste into L0_PRODUCTS in sentinel1_l0_to_slc_ondemand.ipynb):")
print("L0_PRODUCTS = [")
for name in EW_RAW_SCENES:
    print(f'    "{name}",')
print("]")